# Enzyme Kinetics I --- Instructor Solutions

This is the **instructor answer key** for the three exercises in
[`notebooks/03_enzyme_kinetics_I.ipynb`](../notebooks/03_enzyme_kinetics_I.ipynb). For each exercise it gives:

1. a full derivation or worked reasoning (not just the answer),
2. the completed code, runnable end-to-end, and
3. **teaching notes**: what a fully-correct submission should show, common
   student mistakes, and a talking point/extension if there's time.

**Not for distribution to students before the exercise deadline.** This notebook
is excluded from the published Quarto site (see `_quarto.yml`'s `!instructor/`
render rule) but is still committed to the repo history like any other file.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def enzyme_full(t, y, k1, km1, kcat):
    E, S, ES, P = y
    v_f = k1 * E * S
    v_r = km1 * ES
    v_cat = kcat * ES
    dE = -v_f + v_r + v_cat
    dS = -v_f + v_r
    dES = v_f - v_r - v_cat
    dP = v_cat
    return [dE, dS, dES, dP]

def enzyme_fixed_S(t, y, k1, km1, kcat, S_val):
    E, ES = y
    v_f = k1 * E * S_val
    v_r = km1 * ES
    v_cat = kcat * ES
    return [-v_f + v_r + v_cat, v_f - v_r - v_cat]

def initial_rate(k1, km1, kcat, Etot, S0_val, t_ss=1.0):
    s = solve_ivp(enzyme_fixed_S, (0, t_ss), [Etot, 0.0],
                   args=(k1, km1, kcat, S0_val), rtol=1e-10, atol=1e-14)
    return kcat * s.y[1, -1]

k1, km1, kcat = 100.0, 50.0, 10.0
Etot = 1.0
S0 = 100.0
KM = (km1 + kcat) / k1
Vmax = kcat * Etot
t_eval = np.linspace(0, 15, 600)
print(f"KM = {KM:.3f} uM, Vmax = {Vmax:.3f} uM/s")


KM = 0.600 uM, Vmax = 10.000 uM/s


## Exercise 1: QSSA Accuracy vs. $E_{tot}$

### Derivation

The quasi-steady-state approximation for $[ES]$ assumes $[ES]$ equilibrates fast
compared to how quickly $[S]$ is depleted. Segel's parameter for how good that
assumption is is

$$
\varepsilon = \frac{E_{tot}}{K_M+S_0},
$$

with QSSA accurate when $\varepsilon\ll1$. In the base example, $\varepsilon =
1/(0.6+100)\approx0.0099$ — very small, hence the tight agreement already seen.
Raising $E_{tot}$ from 1 to 20 raises $\varepsilon$ to $20/100.6\approx0.199$, a
**~20x increase in $\varepsilon$**. Segel & Slemrod's singular-perturbation result
is that the QSSA error is $O(\varepsilon)$ — an upper-bound *order*, not a tight
proportionality — so the post-transient error should visibly worsen, but there's no
guarantee it worsens by the same factor $\varepsilon$ did; the actual run below
shows a real but sub-proportional increase (a few-fold, not ~20x), which is itself
worth having students notice rather than expecting a clean linear match. Note the
fast-transient timescale $t_{fast}=5/(k_1S_0+k_{-1}+k_{cat})$ **doesn't depend on
$E_{tot}$ at all** — only the size of the residual error *after* that transient
changes.


In [2]:
Etot_new = 20.0

sol_orig = solve_ivp(enzyme_full, (0, 15), [Etot, S0, 0.0, 0.0],
                      t_eval=t_eval, args=(k1, km1, kcat), rtol=1e-8, atol=1e-10)
sol_new = solve_ivp(enzyme_full, (0, 15), [Etot_new, S0, 0.0, 0.0],
                     t_eval=t_eval, args=(k1, km1, kcat), rtol=1e-8, atol=1e-10)

t_fast = 5.0 / (k1 * S0 + km1 + kcat)   # independent of Etot
after_transient = t_eval > 10 * t_fast

for label, sol, Et in [("Etot=1 (original)", sol_orig, Etot), ("Etot=20", sol_new, Etot_new)]:
    _, S, ES, _ = sol.y
    ES_qssa = Et * S / (KM + S)
    max_err_after = np.max(np.abs(ES[after_transient] - ES_qssa[after_transient])) / Et
    epsilon = Et / (KM + S0)
    print(f"{label}: epsilon=Etot/(KM+S0)={epsilon:.4f}, max_err_after={max_err_after:.4f}")


Etot=1 (original): epsilon=Etot/(KM+S0)=0.0099, max_err_after=0.0166
Etot=20: epsilon=Etot/(KM+S0)=0.1988, max_err_after=0.0404


### Teaching notes

**What a correct submission looks like:** `max_err_after` for `Etot=20`
(≈0.040) visibly larger than for `Etot=1` (≈0.017) — roughly 2-3x, not the ~20x
growth seen in $\varepsilon$ itself — with a correct explanation citing $E_{tot}$
becoming comparable to $K_M+S_0$ (the assumption being *strained*, not destroyed)
rather than "the code broke." A student who expects the error to scale linearly
with $\varepsilon$ and is surprised it doesn't is engaging with the material
correctly; $O(\varepsilon)$ is an asymptotic bound, not an equality.

**Common mistakes:**

- Recomputing `KM` and `Vmax` using `Etot_new` — `KM` doesn't depend on $E_{tot}$ at
  all (it's a ratio of rate constants only), and while `Vmax=kcat*Etot` **does**
  scale with $E_{tot}$, plugging the wrong `Etot` into the *QSSA formula for
  $[ES]$* — which is `Etot*S/(KM+S)` — is the actual bug to watch for, since it's
  easy to reuse the original `Etot` variable there by accident.
- Attributing the larger error to a "worse fit" of the QSSA algebraically, rather
  than to violating its underlying assumption ($E_{tot}\ll K_M+S_0$).

**Extension, if there's time:** ask at what $E_{tot}$ the QSSA error would become
"unacceptable" for a given tolerance (e.g. 1%) — this is exactly the kind of
back-of-envelope check ($\varepsilon\lesssim0.01$) worth doing *before* trusting a
QSSA-based rate law in a larger network model.


## Exercise 2: $v_0$ at $S=K_M$

### Derivation

By construction $v_0(K_M) = V_{max}K_M/(K_M+K_M) = V_{max}/2$ exactly. Since
`S_values = np.logspace(-2, 2, 25)` is a fixed log-spaced grid, no sample lands
exactly on $K_M=0.6$; the nearest sample is off by some $\Delta S$, and the
resulting deviation in $v_0$ can be predicted from the local slope of the MM curve
at $S=K_M$:

$$
\left.\frac{dv_0}{dS}\right|_{S=K_M} = \frac{V_{max}K_M}{(K_M+S)^2}\bigg|_{S=K_M}
= \frac{V_{max}}{4K_M}.
$$


In [3]:
S_values = np.logspace(-2, 2, 25)
v_sim = np.array([initial_rate(k1, km1, kcat, Etot, s) for s in S_values])

idx = np.argmin(np.abs(S_values - KM))
S_closest = S_values[idx]
delta_S = S_closest - KM
predicted_dev = (Vmax / (4 * KM)) * delta_S

print(f"Closest grid point to KM={KM:.3f}: S={S_closest:.4f} (delta={delta_S:+.4f})")
print(f"v_sim there = {v_sim[idx]:.4f} uM/s, Vmax/2 = {Vmax / 2:.4f} uM/s, "
      f"actual deviation = {v_sim[idx] - Vmax / 2:+.4f}")
print(f"Predicted deviation from local slope Vmax/(4*KM)*delta_S = {predicted_dev:+.4f}")


Closest grid point to KM=0.600: S=0.6813 (delta=+0.0813)
v_sim there = 5.3172 uM/s, Vmax/2 = 5.0000 uM/s, actual deviation = +0.3172
Predicted deviation from local slope Vmax/(4*KM)*delta_S = +0.3387


### Teaching notes

**What a correct submission looks like:** `v_sim[idx]` close to but not exactly
`Vmax/2`, with the gap explained by the grid not landing exactly on $K_M$ (not
treated as an error in the simulation). The stronger version of this exercise —
shown above — predicts the size of that gap analytically from the local slope and
confirms it matches the observed deviation.

**Common mistakes:**

- Expecting `v_sim[idx]` to equal `Vmax/2` exactly and treating any difference as a
  bug — the MM relationship $v_0(K_M)=V_{max}/2$ is only exact at $S=K_M$ precisely,
  not at "the nearest sampled point."
- Using `np.argmin(np.abs(S_values - KM))` correctly but then comparing against the
  wrong `S_values` entry when reporting `v_sim` (an off-by-one from mismatched
  indexing between `S_values` and `v_sim`, which were built as separate arrays).

**Extension, if there's time:** ask what grid density (`np.logspace(..., N)`) would
be needed to get within 1% of $V_{max}/2$ at $S=K_M$ purely from grid resolution —
ties log-spaced sampling density directly to numerical accuracy near a
point of interest.


## Exercise 3: $K_M$, $V_{max}$ vs. $k_{cat}$

### Derivation

$$
V_{max}=k_{cat}E_{tot}, \qquad K_M=\frac{k_{-1}+k_{cat}}{k_1}.
$$

$V_{max}$ scales **exactly linearly** with $k_{cat}$ — halving $k_{cat}$ halves
$V_{max}$, no exceptions. $K_M$ does **not**: here $k_{-1}=50\gg k_{cat}=10$, so
$K_M\approx k_{-1}/k_1$ is dominated by the (unchanged) dissociation rate constant,
and halving $k_{cat}$ only nudges $K_M$ down slightly rather than by anything close
to 50%. This is the "rapid-equilibrium" limit ($k_{-1}\gg k_{cat}$), where $K_M$
behaves almost like a true dissociation constant (binding affinity) and is nearly
insensitive to the catalytic step.


In [4]:
kcat_new = kcat / 2
Vmax_new = kcat_new * Etot
KM_new = (km1 + kcat_new) / k1

print(f"Original: KM={KM:.4f} uM, Vmax={Vmax:.4f} uM/s")
print(f"kcat/2:   KM={KM_new:.4f} uM, Vmax={Vmax_new:.4f} uM/s")
print(f"Vmax ratio (new/old) = {Vmax_new / Vmax:.3f}  (expect exactly 0.5)")
print(f"KM ratio (new/old)   = {KM_new / KM:.3f}  (expect close to 1, since km1 >> kcat)")


Original: KM=0.6000 uM, Vmax=10.0000 uM/s
kcat/2:   KM=0.5500 uM, Vmax=5.0000 uM/s
Vmax ratio (new/old) = 0.500  (expect exactly 0.5)
KM ratio (new/old)   = 0.917  (expect close to 1, since km1 >> kcat)


### Teaching notes

**What a correct submission looks like:** `Vmax` ratio is exactly `0.5`; `KM` ratio
is close to `1` (a small decrease, not a 50% drop) — and, crucially, an explanation
of *why* they respond so differently, not just the two numbers.

**Common mistakes:**

- Assuming $K_M$ must also halve "because $k_{cat}$ halved and $K_M$'s formula
  contains $k_{cat}$" — true that it contains $k_{cat}$, but only as one term in a
  sum dominated by $k_{-1}$ here. Whether $K_M$ tracks $k_{cat}$ closely depends on
  the ratio $k_{-1}/k_{cat}$, not on the mere presence of $k_{cat}$ in the formula.
- Reporting $K_M$ and $V_{max}$ without connecting back to what they mean
  biologically: this exercise is really asking "does making the enzyme a slower
  catalyst change how tightly it appears to bind substrate?" — and the answer here
  is essentially no, because binding/release ($k_1,k_{-1}$) is untouched.

**Extension, if there's time:** ask what would happen instead if $k_{-1}\ll k_{cat}$
(the opposite, "fast turnover" limit) — there $K_M\approx k_{cat}/k_1$ and *would*
track $k_{cat}$ almost linearly, the opposite qualitative behavior from this
exercise's regime.


## Grading rubric summary

| Exercise | Full marks requires | Partial credit for |
|---|---|---|
| 1. QSSA vs. $E_{tot}$ | Correct re-run with `Etot=20`; explanation via $\varepsilon=E_{tot}/(K_M+S_0)$, not a vague "less accurate" | Correct numbers, no explanation of the underlying assumption being violated |
| 2. $v_0$ at $S=K_M$ | Nearest-grid-point value found and compared to $V_{max}/2$; gap explained (ideally quantified via local slope) | Correct nearest point found, gap noted but not explained |
| 3. $K_M$, $V_{max}$ vs. $k_{cat}$ | Both recomputed correctly; correct qualitative explanation ($k_{-1}\gg k_{cat}$ regime) | Correct numbers, no explanation of the asymmetric response |

**Reference:** Ingalls, B. P. (2013). *Mathematical Modeling in Systems Biology: An
Introduction*. MIT Press. Official PDF (with solutions):
<https://www.math.uwaterloo.ca/~bingalls/MMSB/MMSB_w_solutions.pdf>
